# SLA Extraction RAG Pipeline — Quickstart

This notebook demonstrates how to:
1. Download sample CUAD contracts
2. Run the SLA extraction pipeline
3. Query results for monetary penalties

## Setup

First, ensure you have the required dependencies installed:

```bash
pip install -r requirements.txt
```

And configure your environment:

```bash
echo "DEEPSEEK_API_KEY=sk-your-key-here" > .env
echo "HF_TOKEN=hf_your-token-here" >> .env  # optional
```

In [ ]:
import os
import sys
import sqlite3
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Add project to path
sys.path.insert(0, str(Path.cwd()))

from scripts._batch import run_batch
from pipeline.output import init_db

print("✅ Dependencies loaded")

## Step 1: Download Sample Contracts

Let's download 5 sample contracts from the CUAD dataset.

In [ ]:
# Download 5 sample CUAD contracts
os.system("python scripts/download_cuad.py --max 5")
print("✅ Sample contracts downloaded to data/cuad_raw/")

## Step 2: Run Pipeline

Process the 5 contracts through the full extraction pipeline.

In [ ]:
# Initialize database
init_db(reset=True)

# Run batch pipeline on 5 contracts
data_dir = Path("data/cuad_raw")
run_batch(data_dir, max_contracts=5)

print("✅ Pipeline complete. Results saved to output/")

## Step 3: Inspect Results

Load and analyze the extracted SLA data.

In [ ]:
# Load results from SQLite
conn = sqlite3.connect("output/results.db")
df = pd.read_sql("SELECT * FROM sla_results", conn)

print(f"Processed {len(df)} contracts")
print(f"\nColumns: {list(df.columns)}")
df.head()

## Step 4: Query Monetary Penalties

Find contracts with real cash penalties (not just service credits).

In [ ]:
# Contracts with real monetary penalties
monetary = pd.read_sql(
    """SELECT contract_id, penalty_has_monetary, penalty_max_amount, penalty_currency
       FROM sla_results WHERE penalty_has_monetary = 1""",
    conn
)

print(f"Contracts with monetary penalties: {len(monetary)}")
print(monetary)

## Step 5: Analyze Late Payment Penalties

Which contracts have late payment interest clauses?

In [ ]:
# Late payment penalties
late_pay = pd.read_sql(
    """SELECT contract_id, penalty_late_payment
       FROM sla_results WHERE penalty_late_payment IS NOT NULL""",
    conn
)

print(f"Contracts with late payment penalties: {len(late_pay)}")
print(late_pay)

## Step 6: Field Coverage Summary

In [ ]:
# Summary of which fields are populated
coverage = {
    'termination_clause': (df['termination_clause'].notna().sum(), len(df)),
    'governing_law': (df['governing_law'].notna().sum(), len(df)),
    'liability_cap': (df['liability_cap'].notna().sum(), len(df)),
    'penalty_has_monetary': (df['penalty_has_monetary'].notna().sum(), len(df)),
    'penalty_late_payment': (df['penalty_late_payment'].notna().sum(), len(df)),
    'dispute_resolution': (df['dispute_resolution'].notna().sum(), len(df)),
}

coverage_df = pd.DataFrame([
    {'field': k, 'populated': v[0], 'total': v[1], 'coverage_%': round(100*v[0]/v[1], 1)}
    for k, v in coverage.items()
])

print("Field Coverage Summary")
print(coverage_df.to_string(index=False))

## Step 7: Inspect Raw Extraction for One Contract

View the full raw response from the LLM for debugging.

In [ ]:
import json

# Get the first contract
first_contract = df.iloc[0]
contract_id = first_contract['contract_id']

print(f"Contract: {contract_id}")
print(f"Status: {first_contract['status']}")

if first_contract['raw_response']:
    raw = json.loads(first_contract['raw_response'])
    print("\nRaw LLM Response:")
    for key, value in raw.items():
        if value:
            print(f"  {key}: {value[:100]}..." if len(str(value)) > 100 else f"  {key}: {value}")

## Next Steps

- **Scale up:** Download all 510 CUAD contracts with `python scripts/download_cuad.py` (no `--max`)
- **Full run:** Process all with `python scripts/run_pipeline.py data/cuad_raw`
- **Evaluate:** Run evals with `python -m evals.eval_runner`
- **Customize:** Edit `models/prompts.py` to tune the extraction prompt
- **Query:** Use pandas/SQL to analyze the results further

See [docs/ARCHITECTURE.md](../docs/ARCHITECTURE.md) for detailed pipeline information and [docs/COSTS.md](../docs/COSTS.md) for cost analysis.